
HomeGuard Security System Simulator
Author: [Markus von Aschoff]
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.



In [2]:
import random
from datetime import datetime

# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

# Current system state
current_mode = "AWAY"

def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """
    Creates a sensor data structure.

    Parameters:
    - sensor_id: Unique identifier for the sensor
    - location: Where the sensor is located (e.g., "Living Room")
    - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
    - threshold: Optional threshold value for the sensor

    Returns:
    - A dictionary representing the sensor
    """
    sensor = {
        'sensor_id': sensor_id,
        'location': location,
        'type': sensor_type,
        'threshold': threshold
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    """
    Creates an alert data structure.

    Parameters:
    - severity: Alert severity level (LOW, MEDIUM, HIGH, CRITICAL)
    - message: Description of the alert
    - sensor_id: ID of the sensor that triggered the alert
    - timestamp: When the alert was triggered

    Returns:
    - A dictionary representing the alert
    """
    alert = {
        'severity': severity,
        'message': message,
        'sensor_id': sensor_id,
        'timestamp': timestamp

    }
    return alert



# Initialize sensors for the Peterson home
sensors = [
    create_sensor('1', 'Living Room', 'motion'), #motion for the living room
    create_sensor('2', 'Kitchen', 'temperature'), 
    create_sensor('3', 'Front Door', 'door'), #door for the front door
    create_sensor('4', 'Bedroom', 'smoke')  #smoke for the bedroom
]

print(f"Initialized {len(sensors)} sensors")

for sensor in sensors:
    print(f"  - {sensor['sensor_id']}: {sensor['location']} ({sensor['type']})")


Initialized 4 sensors
  - 1: Living Room (motion)
  - 2: Kitchen (temperature)
  - 3: Front Door (door)
  - 4: Bedroom (smoke)


In [3]:
def is_abnormal_reading(sensor, reading_value):
    """
    Checks if a sensor reading is abnormal based on sensor type and thresholds.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    
    Returns:
    - True if the reading is abnormal, False otherwise
    """
    sensor_type = sensor["type"]
    
    # YOUR CODE HERE: Add if/else logic to check for abnormal readings
    # For temperature sensors: check if below 35°F or above 95°F
    
    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value == True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    else:
        return False
    
     
def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)
    
    Returns:
    - True if a security alert should be triggered, False otherwise
    """
    if not is_abnormal_reading(sensor, reading_value):
      return False
 
    if system_mode == "AWAY":
        # Every abnormal reading is a security concern when no one is home
      return True
 
    elif system_mode == "SLEEP":
        # At night, motion and smoke warrant an alert
      return sensor["type"] in ["motion", "smoke"]
 
    elif system_mode == "HOME":
        # Only smoke is always a security concern when people are home
      return sensor["type"] == "smoke"
    return False

In [4]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [5]:
def generate_reading(sensor):
    """
    Generates a realistic reading for a sensor.
    
    Parameters:
    - sensor: Sensor dictionary
    
    Returns:
    - A realistic reading value based on sensor type
    """    
    # YOUR CODE HERE: Generate appropriate readings based on sensor type
    # Use random values to simulate real sensor data
    # - Temperature: random value between 30-100°F
    # - Motion: random boolean (True/False)
    # - Door: random choice between "OPEN" and "CLOSED"
    # - Smoke: random choice between "CLEAR" and "DETECTED"
   
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        return random.randint(30, 100)
 
    elif sensor_type == "motion":
        return random.choice([True, False])
 
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
 
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
 
    else:
        return None
    


 
 
def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)
    """
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_id = sensor["sensor_id"]
  

    # YOUR CODE HERE: Check for different types of alerts
    # 1. Check for security alerts (motion/door in AWAY mode)
    # 2. Check for safety alerts (temperature extremes, smoke)
    # 3. Check for comfort notifications (temperature out of range in HOME mode)
    
   
 # ── Security alerts ──────────────────────────────────────────────
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        if sensor["type"] == "motion":
            severity = "HIGH"
            message  = f"SECURITY: Motion detected in {sensor['location']} while in {system_mode} mode!"
        elif sensor["type"] == "door":
            severity = "HIGH"
            message  = f"SECURITY: {sensor['location']} opened while in {system_mode} mode!"
        elif sensor["type"] == "smoke":
            severity = "CRITICAL"
            message  = f"FIRE RISK: Smoke detected in {sensor['location']}!"
        else:
            return alerts
 
        alerts.append(create_alert(severity, message, sensor_id, timestamp))
 
    # ── Safety alerts (always active, regardless of mode) ────────────
    if sensor["type"] == "temperature":
        if reading_value < 35:
            alerts.append(create_alert(
                "CRITICAL",
                f"SAFETY: Temperature in {sensor['location']} is {reading_value}°F — frozen pipe risk!",
                sensor_id, timestamp
            ))
        elif reading_value > 95:
            alerts.append(create_alert(
                "HIGH",
                f"SAFETY: Temperature in {sensor['location']} is {reading_value}°F — possible equipment failure!",
                sensor_id, timestamp
            ))
 
    # ── Comfort notifications (only relevant in HOME mode) ───────────
    if system_mode == "HOME" and sensor["type"] == "temperature":
        if 35 <= reading_value < 65:
            alerts.append(create_alert(
                "LOW",
                f"COMFORT: {sensor['location']} is a bit cool at {reading_value}°F.",
                sensor_id, timestamp
            ))
        elif 75 < reading_value <= 95:
            alerts.append(create_alert(
                "LOW",
                f"COMFORT: {sensor['location']} is warm at {reading_value}°F.",
                sensor_id, timestamp
            ))
 
    return alerts

    
    

 
# Feel free to change the function signatures or add helper functions as needed, these are just some suggestions to make the code more friendly! 
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")
 
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [6]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")

#print(alerts)

# Test processing

alerts = []
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])
#print(alerts)

Generated reading for Living Room: False
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!


Step 5: Creating Classes
Objective: Convert the sensor dictionary into a Sensor class with methods.

What to do:

Define a Sensor class with an
__init__
method
Add methods:
read()
,
isAbnormal()
, and
reset()
Update your code to use the Sensor class instead of dictionaries
Explanation: Classes are blueprints for creating objects. They combine data (properties) and functions (methods) into a single unit. This makes your code more organized and easier to work with.

    

In [7]:
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        """
        Initializes a new sensor.
        
        Parameters:
        - sensor_id: Unique identifier for the sensor
        - location: Where the sensor is located
        - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
        - threshold: Optional threshold value for the sensor
        """
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        
    
    def read(self):
        """
        Generates and stores a new reading for this sensor.
        
        Returns:
        - The reading value
        """
        # YOUR CODE HERE: Generate a reading and store it in self.current_value

        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold,
        }
        self.current_value = generate_reading(sensor_dict)
        return self.current_value   
    
        
    def isAbnormal(self):
        """
        Checks if the current reading is abnormal.
        
        Returns:
        - True if the reading is abnormal, False otherwise
        """
        # YOUR CODE HERE: Use the is_abnormal_reading function logic
        if self.current_value is None:
            return False
        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold,
        }
        return is_abnormal_reading(sensor_dict, self.current_value) 


        
    
    def reset(self):
        """
        Resets the sensor's current reading to None.
        """
        # YOUR CODE HERE: Set current_value to None
        self.current_value = None
    
    def __str__(self):
        """
        Returns a string representation of the sensor.
        """
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
 
# Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

In [8]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 53
Is abnormal: False
Sensor info: TEST_001 (Test Room): 53


Step 6: Integrating Everything
Objective: Combine all components into a complete simulator that runs continuously.

What to do:

Create a main simulation loop
Integrate all your functions and classes
Format the output to match the sample

In [9]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "sensor_id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable (optional)
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation
    run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 15:55:37
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [15:55:37] Sending notification to homeowner...
[READING] Kitchen Temperature: 50°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 15:55:37
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 52°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: FIRE RISK: Smoke detected in Bedroom!
[LOG] [15:55:37] Sending notification to homeowner...

Time: 15:55:38
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 75°F (Normal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [15:55:38] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Simulation complete!
